In [183]:

from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

In [139]:
from motor_ingesta.motor_ingesta import MotorIngesta
from pyspark.sql import DataFrame as DF, functions as F

In [143]:
json_path="abfss://datos@masterap001sta.dfs.core.windows.net/2023-01-01.json"
flights_day_df = spark.read.option("multiLine", "true").json(json_path)

In [144]:
flights_day_df

DataFrame[FlightDate: string, flights_list: array<struct<ActualElapsedTime:double,AirTime:double,CRSArrTime:bigint,CRSDepTime:bigint,CRSElapsedTime:double,CancellationCode:string,Cancelled:double,Distance:double,DistanceGroup:bigint,Diverted:double,FirstDepTime:bigint,Flights:double,LongestAddGTime:double,Tail_Number:string,TaxiIn:double,TaxiOut:double,TotalAddGTime:double,WheelsOff:bigint,WheelsOn:bigint,airline_info:struct<DOT_ID_Reporting_Airline:bigint,Flight_Number_Reporting_Airline:bigint,IATA_CODE_Reporting_Airline:string,Reporting_Airline:string>,arrival_performance:struct<ArrDel15:double,ArrDelay:double,ArrDelayMinutes:double,ArrTime:bigint,ArrTimeBlk:string,ArrivalDelayGroups:bigint>,delay_info:struct<CarrierDelay:double,LateAircraftDelay:double,NASDelay:double,SecurityDelay:double,WeatherDelay:double>,departure_performance:struct<DepDel15:double,DepDelay:double,DepDelayMinutes:double,DepTime:bigint,DepTimeBlk:string,DepartureDelayGroups:bigint>,destination_info:struct<Dest:s

In [146]:
def aplana_df(df: DF) -> DF:
    """
    Aplana un DataFrame de Spark que tenga columnas de tipo array y de tipo estructura.

    :param df: DataFrame de Spark que contiene columnas de tipo array o columnas de tipo estructura, incluyendo
               cualquier nivel de anidamiento y también arrays de estructuras. Asumimos que los nombres de los
               campos anidados son todos distintos entre sí, y no van a coincidir cuando sean aplanados.
    :return: DataFrame de Spark donde todas las columnas de tipo array han sido explotadas y las estructuras
             han sido aplanadas recursivamente.
    """
    to_select = []
    schema = df.schema.jsonValue()
    fields = schema["fields"]
    recurse = False

    for f in fields:
        if f["type"].__class__.__name__ != "dict":
            to_select.append(f["name"])
        else:
            if f["type"]["type"] == "array":
                to_select.append(F.explode(f["name"]).alias(f["name"]))
                recurse = True
            elif f["type"]["type"] == "struct":
                to_select.append(f"{f['name']}.*")
                recurse = True

    new_df = df.select(*to_select)
    return MotorIngesta.aplana_df(new_df) if recurse else new_df

In [147]:
aplanado_df = aplana_df(flights_day_df)


In [148]:
aplanado_df

DataFrame[FlightDate: string, ActualElapsedTime: double, AirTime: double, CRSArrTime: bigint, CRSDepTime: bigint, CRSElapsedTime: double, CancellationCode: string, Cancelled: double, Distance: double, DistanceGroup: bigint, Diverted: double, FirstDepTime: bigint, Flights: double, LongestAddGTime: double, Tail_Number: string, TaxiIn: double, TaxiOut: double, TotalAddGTime: double, WheelsOff: bigint, WheelsOn: bigint, DOT_ID_Reporting_Airline: bigint, Flight_Number_Reporting_Airline: bigint, IATA_CODE_Reporting_Airline: string, Reporting_Airline: string, ArrDel15: double, ArrDelay: double, ArrDelayMinutes: double, ArrTime: bigint, ArrTimeBlk: string, ArrivalDelayGroups: bigint, CarrierDelay: double, LateAircraftDelay: double, NASDelay: double, SecurityDelay: double, WeatherDelay: double, DepDel15: double, DepDelay: double, DepDelayMinutes: double, DepTime: bigint, DepTimeBlk: string, DepartureDelayGroups: bigint, Dest: string, DestAirportID: bigint, DestAirportSeqID: bigint, DestCityMark

In [149]:
import json
from pathlib import Path


root_path = Path.cwd().parent
config_file= str(root_path/"config"/"config.json")


In [150]:
print(config_file)

C:\repos\repos_master\repo_spark\spark-tarea-final\config\config.json


In [151]:
with open(config_file, "r") as f:
    config = json.load(f)

In [152]:
lista_obj_column = [F.col(diccionario["name"]).cast(diccionario["type"]).alias(diccionario["name"], metadata={"comment": diccionario["comment"]})
                                  for diccionario in config["data_columns"] ]
resultado_df = aplanado_df.select(*lista_obj_column)

In [153]:
resultado_df.head(5)

[Row(FlightDate=datetime.date(2023, 1, 1), Reporting_Airline='9E', OriginAirportID=12884, Origin='LAN', OriginCityName='Lansing, MI', OriginState='MI', DestAirportID=11433, Dest='DTW', DestCityName='Detroit, MI', DestState='MI', DepDelay=-6, DepTime=1136, ArrDelay=-22, ArrTime=1215, Cancelled=False, Diverted=False, AirTime=21, Distance=74),
 Row(FlightDate=datetime.date(2023, 1, 1), Reporting_Airline='DL', OriginAirportID=13487, Origin='MSP', OriginCityName='Minneapolis, MN', OriginState='MN', DestAirportID=13495, Dest='MSY', DestCityName='New Orleans, LA', DestState='LA', DepDelay=-1, DepTime=1015, ArrDelay=-17, ArrTime=1244, Cancelled=False, Diverted=False, AirTime=133, Distance=1039),
 Row(FlightDate=datetime.date(2023, 1, 1), Reporting_Airline='AS', OriginAirportID=14747, Origin='SEA', OriginCityName='Seattle, WA', OriginState='WA', DestAirportID=14679, Dest='SAN', DestCityName='San Diego, CA', DestState='CA', DepDelay=1, DepTime=1946, ArrDelay=8, ArrTime=2241, Cancelled=False, Div

In [155]:
from motor_ingesta.agregaciones import aniade_hora_utc, aniade_intervalos_por_aeropuerto
import pandas as pd

In [156]:
path_timezones =  Path.cwd() / "resources" / "timezones.csv"
timezones_pd = pd.read_csv(path_timezones)
timezones_df = spark.createDataFrame(timezones_pd)

In [157]:
timezones_df.head(5)

[Row(iata_code='AAA', iana_tz='Pacific/Tahiti', windows_tz='Hawaiian Standard Time'),
 Row(iata_code='AAB', iana_tz='Australia/Brisbane', windows_tz='E. Australia Standard Time'),
 Row(iata_code='AAC', iana_tz='Africa/Cairo', windows_tz='Egypt Standard Time'),
 Row(iata_code='AAD', iana_tz='Africa/Mogadishu', windows_tz='E. Africa Standard Time'),
 Row(iata_code='AAE', iana_tz='Africa/Algiers', windows_tz='W. Central Africa Standard Time')]

In [159]:
df_with_tz = resultado_df.join(timezones_df, resultado_df.Origin == timezones_df.iata_code, "left")

In [160]:
df_with_tz.head(1)

[Row(FlightDate=datetime.date(2023, 1, 1), Reporting_Airline='9E', OriginAirportID=12884, Origin='LAN', OriginCityName='Lansing, MI', OriginState='MI', DestAirportID=11433, Dest='DTW', DestCityName='Detroit, MI', DestState='MI', DepDelay=-6, DepTime=1136, ArrDelay=-22, ArrTime=1215, Cancelled=False, Diverted=False, AirTime=21, Distance=74, iata_code='LAN', iana_tz='America/Detroit', windows_tz='Eastern Standard Time')]

In [162]:
df_with_flight_time = df_with_tz.withColumn(
        "castedHour", F.lpad(F.col("DepTime").cast("string"), 4, "0")
    ).withColumn(
        "FlightTime",
        F.concat(
            F.col("FlightDate").cast("string"),
            F.lit(" "),
            F.col("castedHour").substr(1, 2),
            F.lit(":"),
            F.col("castedHour").substr(3, 2)
        ).cast("timestamp")
    )

In [163]:
df_with_flight_time.show(1)

+----------+-----------------+---------------+------+--------------+-----------+-------------+----+------------+---------+--------+-------+--------+-------+---------+--------+-------+--------+---------+---------------+--------------------+----------+-------------------+
|FlightDate|Reporting_Airline|OriginAirportID|Origin|OriginCityName|OriginState|DestAirportID|Dest|DestCityName|DestState|DepDelay|DepTime|ArrDelay|ArrTime|Cancelled|Diverted|AirTime|Distance|iata_code|        iana_tz|          windows_tz|castedHour|         FlightTime|
+----------+-----------------+---------------+------+--------------+-----------+-------------+----+------------+---------+--------+-------+--------+-------+---------+--------+-------+--------+---------+---------------+--------------------+----------+-------------------+
|2023-01-01|               9E|          12884|   LAN|   Lansing, MI|         MI|        11433| DTW| Detroit, MI|       MI|      -6|   1136|     -22|   1215|    false|   false|     21|    

In [165]:
df_with_flight_time = df_with_flight_time.withColumn(
        "FlightTime", F.to_utc_timestamp(F.col("FlightTime"), F.col("iana_tz"))
    )

In [166]:
df_with_flight_time.show(1)

+----------+-----------------+---------------+------+--------------+-----------+-------------+----+------------+---------+--------+-------+--------+-------+---------+--------+-------+--------+---------+---------------+--------------------+----------+-------------------+
|FlightDate|Reporting_Airline|OriginAirportID|Origin|OriginCityName|OriginState|DestAirportID|Dest|DestCityName|DestState|DepDelay|DepTime|ArrDelay|ArrTime|Cancelled|Diverted|AirTime|Distance|iata_code|        iana_tz|          windows_tz|castedHour|         FlightTime|
+----------+-----------------+---------------+------+--------------+-----------+-------------+----+------------+---------+--------+-------+--------+-------+---------+--------+-------+--------+---------+---------------+--------------------+----------+-------------------+
|2023-01-01|               9E|          12884|   LAN|   Lansing, MI|         MI|        11433| DTW| Detroit, MI|       MI|      -6|   1136|     -22|   1215|    false|   false|     21|    

In [168]:
cols_to_drop = timezones_df.columns + ["castedHour"]

df_with_flight_time= df_with_flight_time.drop(*cols_to_drop)

In [169]:
from datetime import timedelta
from loguru import logger

dia_actual = resultado_df.first().FlightDate
dia_previo = dia_actual - timedelta(days=1)

In [170]:
print(dia_previo, dia_actual)

2022-12-31 2023-01-01


In [171]:
try:
    flights_previo = spark.read.table(config["output_table"]).where(F.col("FlightDate") == dia_previo)
    flights_previo.limit(1).collect()
    logger.info(f"Leída partición del día {dia_previo} con éxito")
except Exception as e:
    logger.info(f"No se han podido leer datos del día {dia_previo}: {str(e)}")
    flights_previo = None

2026-04-06 19:44:36.231 | INFO     | __main__:<module>:6 - No se han podido leer datos del día 2022-12-31: [TABLE_OR_VIEW_NOT_FOUND] The table or view `default`.`flights` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01;
'GlobalLimit 1
+- 'LocalLimit 1
   +- 'Filter '`==`('FlightDate, 2022-12-31)
      +- 'UnresolvedRelation [default, flights], [], false



In [172]:
print(spark.catalog.currentCatalog())
print(spark.catalog.currentDatabase())
print(spark.catalog.listTables())

masterap001dbrpremiun
default
[]


In [174]:
if flights_previo:
    # añadir columnas a F.lit(None) haciendo cast al tipo adecuado de cada una, y unirlo con flights_previo.
    # OJO: hacer select(flights_previo.columns) para tenerlas en el mismo orden antes de
    # la unión, ya que la columna de partición se había ido al final al escribir

    cols_vuelo_siguiente = ["FlightTime_next", "Airline_next", "diff_next"]
    df_hoy_preparado = df_with_flight_time
    for col in cols_vuelo_siguiente:
        df_hoy_preparado = df_hoy_preparado.withColumn(col, F.lit(None))

    df_unido = flights_previo.select(df_hoy_preparado.columns).union(df_hoy_preparado)

    # Spark no permite escribir en la misma tabla de la que estamos leyendo. Por eso salvamos
    df_unido.write.mode("overwrite").saveAsTable("tabla_provisional")
    df_unido = spark.read.table("tabla_provisional")

else:
    df_unido = df_with_flight_time

In [176]:
df_with_next_flight = aniade_intervalos_por_aeropuerto(df_unido)

In [177]:
df_with_next_flight.show(1)

+----------+-----------------+---------------+------+--------------------+-----------+-------------+----+------------+---------+--------+-------+--------+-------+---------+--------+-------+--------+-------------------+-------------------+------------+---------+
|FlightDate|Reporting_Airline|OriginAirportID|Origin|      OriginCityName|OriginState|DestAirportID|Dest|DestCityName|DestState|DepDelay|DepTime|ArrDelay|ArrTime|Cancelled|Diverted|AirTime|Distance|         FlightTime|    FlightTime_next|Airline_next|diff_next|
+----------+-----------------+---------------+------+--------------------+-----------+-------------+----+------------+---------+--------+-------+--------+-------+---------+--------+-------+--------+-------------------+-------------------+------------+---------+
|2023-01-01|               9E|          10135|   ABE|Allentown/Bethleh...|         PA|        10397| ATL| Atlanta, GA|       GA|       1|    601|      -9|    816|    false|   false|    113|     692|2023-01-01 16:01

In [178]:
df_with_next_flight\
        .coalesce(config.get("output_partitions", 1))\
        .write.mode("overwrite").option("partitionOverwriteMode", "dynamic")\
        .partitionBy("FlightDate")\
        .saveAsTable(config["output_table"])

logger.info(f"Procesamiento completado para el día {dia_actual}")


# Borrar la tabla provisional si la hubiéramos creado
spark.sql("DROP TABLE IF EXISTS tabla_provisional")



2026-04-06 19:46:27.347 | INFO     | __main__:<module>:7 - Procesamiento completado para el día 2023-01-01


DataFrame[]

In [184]:
spark.sql("SHOW STORAGE CREDENTIALS").show()

+--------------------+-------+
|                name|comment|
+--------------------+-------+
|masterap001dbrpre...|   NULL|
+--------------------+-------+



In [185]:
nombre_location = "external_location_flights"
url_azure = "abfss://datos@masterap001sta.dfs.core.windows.net/"
credencial = "masterap001dbrpremiun" # El resultado del paso 1

try:
    spark.sql(f"""
        CREATE EXTERNAL LOCATION IF NOT EXISTS {nombre_location}
        URL '{url_azure}'
        WITH (STORAGE CREDENTIAL `{credencial}`)
    """)
    print(f"¡Éxito! Localización externa '{nombre_location}' creada.")
except Exception as e:
    print(f"Error al crear la localización: {e}")

Error al crear la localización: (com.databricks.sql.managedcatalog.acl.UnauthorizedAccessException) PERMISSION_DENIED: The contributor role on the storage account is not set or Managed Identity does not have READ, LIST, WRITE, DELETE permissions on url abfss://datos@masterap001sta.dfs.core.windows.net/. Please contact your account admin to update the storage credential. Operation failed: "This request is not authorized to perform this operation using this permission.", 403, HEAD, , https://masterap001sta.dfs.core.windows.net/datos/validate_credential_2026-04-06-19-51-03?upn=false&action=getStatus&timeout=90
